<a id="top"></a>

# Demo: roles, and where content belongs

```yaml
title: "Demo: roles, and where content belongs"
keywords: roles, system message, user message, assistant, messages list, multi-turn, cost staircase, teacher demo
estimated duration: 18
```

> **Lesson:** L02. Teacher demo — Demo 1 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L02/demos_or_activities.md). Makes **live** Claude calls through the `potato_llm` seam — set `ANTHROPIC_API_KEY` (copy `.env.example` to `.env`) before running. A real model **won't** reproduce every effect on cue; the prose frames what to *watch for*, and that variance is itself part of the lesson. Clear outputs before committing.
>
> Sticky note for the whole demo: **system = always-true, user = per-call.**

## Contents

- [1. Setup](#1-setup)
- [2. Where the same content lands changes the answer](#2-where-the-same-content-lands-changes-the-answer)
- [3. The messages list grows, and you own it](#3-the-messages-list-grows-and-you-own-it)
- [4. Takeaways](#4-takeaways)

## 1. Setup

One live client, used for every variant below. We change only *where* the policy and the patient data sit in the messages list, and watch how the reply shifts.

In [ ]:
from fluffy_potato_curriculum.potato_llm import AnthropicClient, Message, PotatoLLMClient

# Live client. The key is read through fluffy_potato_curriculum.common.config
# (a pydantic-settings layer over the environment / .env); constructing the client
# raises a clear error if ANTHROPIC_API_KEY is missing, rather than failing mid-call.
client: PotatoLLMClient = AnthropicClient()
print(f"setup ready (live client: {client.model})")

[↑ Back to top](#top)

## 2. Where the same content lands changes the answer

Three variants of the **same task** (a clinic triage assistant). Only the *placement* of the policy and the patient data changes.

In [ ]:
POLICY = (
    "You are a triage assistant for a small clinic. Always respond in two short "
    "paragraphs: a recommended next step, then any cautions. Never give specific "
    "dosage advice."
)
QUESTION = "My head has been hurting for three days. What should I do?"


def show(label: str, messages: list[Message]) -> None:
    """Send a message list to the live model and print the reply."""
    reply = client.chat(messages, max_tokens=200, temperature=0.0)
    print(f"=== {label} ===")
    print(reply.text)
    print(f"(input ~{reply.usage.input_tokens} tok)\n")


# Variant 1: policy in `system` (always-true), request in `user` (per-call).
show(
    "variant 1 — system carries policy, user carries request",
    [Message.system(POLICY), Message.user(QUESTION)],
)

# Variant 2: everything shoved into the user message.
show(
    "variant 2 — everything in the user message",
    [Message.user(POLICY + " " + QUESTION)],
)

Variant 1 anchors the policy in `system`, where it applies to *every* call; variant 2 puts the same words in `user`. With a real model the difference is a **tendency, not a guarantee** — variant 2 is more likely to drift from the two-paragraph format, and an unanchored policy degrades faster over a long chat. Run it a few times: that you *can't* count on it is the point — **strongly-weighted ≠ enforced.**

In [ ]:
# Variant 3: per-call patient data parked in the SYSTEM message (the reuse footgun).
stale_system = Message.system(
    "You are a triage assistant. The patient's head has been hurting for three days."
)
show(
    "variant 3 — stale data in system, asking about the head",
    [stale_system, Message.user("What should I do?")],
)

# Now REUSE the same system message for a different patient question.
show(
    "variant 3 reused — same system, a NEW question",
    [stale_system, Message.user("My ankle is swollen. What should I do?")],
)

The `system` message is meant to be **always-true**, so per-call data placed there rides along on *every* later request. A capable model may still answer the new ankle question correctly — but the stale head-pain context is now silently in scope on each call, biasing the reply and inflating input cost. The rule holds whether or not you catch it misbehaving: **per-call data belongs in `user`, not `system`.**

[↑ Back to top](#top)

## 3. The messages list grows, and you own it

The "conversation" is just a Python list of `{role, content}` messages your code re-sends every turn. Watch it grow — and watch the input-token staircase from L01.

In [ ]:
def render(messages: list[Message]) -> str:
    """Render a messages list as readable {role: content} lines."""
    return "\n".join(f"  {m.role:9} | {m.content[:60]}" for m in messages)


conversation: list[Message] = [Message.system(POLICY)]
follow_ups = [
    "My head has been hurting for three days. What should I do?",
    "Should I be worried it is something serious?",
    "What over-the-counter options exist?",
]

for turn, question in enumerate(follow_ups, start=1):
    conversation.append(Message.user(question))
    reply = client.chat(conversation, max_tokens=200, temperature=0.0)
    conversation.append(Message.assistant(reply.text))
    print(
        f"--- after turn {turn}: messages list has {len(conversation)} entries, "
        f"input re-sent ~{reply.usage.input_tokens} tok ---"
    )
    print(render(conversation))
    print()

Two things to say out loud: (1) there is **no hidden server memory** — the whole list is re-sent every turn; (2) because it is re-sent, the **input tokens climb every turn** (the L01 staircase). The `system` message rides along on *every* call, which is exactly why you keep it lean.

[↑ Back to top](#top)

## 4. Takeaways

- **system = always-true, user = per-call.** Durable policy in `system`; this call's data in `user`.
- The system message is **strongly weighted, not enforced** — not a security sandbox.
- The **messages list is yours**: no server memory, and it is re-sent (and re-billed) every turn.
- Per-call data in `system` **poisons reuse**; instructions buried in `user` cost tokens every turn.

[↑ Back to top](#top)